# V9 Phase 1: Download Hieroglyph Dataset

## Goal
Download the HamdiJr/Egyptian_hieroglyphs dataset from HuggingFace, which contains:
- 4,210 labeled hieroglyph images (171 Gardiner classes)
- Lexicon mapping: Gardiner codes → transliteration → English
- Position data for each hieroglyph

## Why This Dataset?
V7 failed to use visual features because we couldn't map transliteration (`nfr`) to Gardiner codes (`F35`).
This dataset provides the missing link:
- **Lexicon.txt**: `D36,N35,D7,;an;beautiful;0.333333;`
- **Images**: `D36.png`, `N35.png`, `D7.png` with Gardiner labels

This allows us to:
1. Map transliteration → Gardiner codes (via lexicon)
2. Extract ResNet-50 features from images (keyed by Gardiner code)
3. Fuse text + visual for each word in our vocabulary

In [ ]:
from pathlib import Path
import subprocess

# Setup paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / 'data/raw'
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project Root: {PROJECT_ROOT}')
print(f'Raw Data Directory: {RAW_DATA_DIR}')

## Download Dataset from HuggingFace

We'll use `huggingface-cli` to download the dataset.

In [ ]:
# Install huggingface_hub if needed
!pip install -q huggingface_hub

In [ ]:
import subprocess
import os

# Use git clone with git-lfs to avoid rate limits
dataset_dir = RAW_DATA_DIR / 'hieroglyph_dataset'

if dataset_dir.exists():
    print(f'Dataset already exists at: {dataset_dir}')
else:
    print('Cloning HamdiJr/Egyptian_hieroglyphs dataset using git-lfs...')
    print('This may take a few minutes (~500MB download)\n')
    
    # Clone the repository
    result = subprocess.run([
        'git', 'clone',
        'https://huggingface.co/datasets/HamdiJr/Egyptian_hieroglyphs',
        str(dataset_dir)
    ], capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f'\n✓ Successfully cloned dataset to: {dataset_dir}')
    else:
        print(f'Error: {result.stderr}')
        raise Exception('Failed to clone dataset')


## Explore Dataset Structure

In [ ]:
import os

# List top-level directories
dataset_dir = RAW_DATA_DIR / 'hieroglyph_dataset'
print('Dataset structure:')
for item in sorted(dataset_dir.rglob('*')):
    if item.is_dir():
        print(f'  📁 {item.relative_to(dataset_dir)}')
    elif item.suffix in ['.txt', '.csv']:
        print(f'  📄 {item.relative_to(dataset_dir)} ({item.stat().st_size} bytes)')

## Load and Inspect Lexicon

In [ ]:
# Find lexicon file
lexicon_path = dataset_dir / 'Dataset/LanguageModel/Lexicon.txt'

if lexicon_path.exists():
    print(f'Found lexicon at: {lexicon_path}')
    
    # Read first 10 lines
    with open(lexicon_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()[:10]
    
    print(f'\nFirst 10 entries:')
    for i, line in enumerate(lines, 1):
        print(f'{i:2d}. {line.strip()}')
else:
    print('Lexicon not found. Searching...')
    for p in dataset_dir.rglob('*.txt'):
        if 'lexicon' in p.name.lower():
            print(f'  Found: {p.relative_to(dataset_dir)}')

## Count Images by Gardiner Code

In [ ]:
from collections import Counter

# Find image directory
manual_dir = dataset_dir / 'Dataset/Manual/Raw'

if manual_dir.exists():
    # Count images by Gardiner code (extracted from filename)
    gardiner_counts = Counter()
    
    for img_path in manual_dir.rglob('*.png'):
        # Filename format: 030000_D35.png
        parts = img_path.stem.split('_')
        if len(parts) == 2:
            gardiner_code = parts[1]
            gardiner_counts[gardiner_code] += 1
    
    print(f'Total images: {sum(gardiner_counts.values())}')
    print(f'Unique Gardiner codes: {len(gardiner_counts)}')
    print(f'\nTop 10 most common:')
    for code, count in gardiner_counts.most_common(10):
        print(f'  {code}: {count} images')
else:
    print('Manual image directory not found. Searching...')
    for p in dataset_dir.rglob('*.png'):
        print(f'  Found image: {p.relative_to(dataset_dir)}')
        break  # Just show first one

## ✅ Download Complete!

**What we got:**
- Images organized by Gardiner code in `Dataset/train/` and `Dataset/test/`
- Filename format: `070029_D21.png` → Gardiner code = `D21`
- We can extract Gardiner codes directly from filenames

**What we already have:**
- `heiro_v6_BERT/data/processed/hieroglyph_lexicon.csv` - Gardiner code mappings

**Next Steps:**
1. Extract ResNet-50 features from these images (keyed by Gardiner code)
2. Use V6 lexicon to map Gardiner codes → Unicode/transliteration
3. Build the fusion pipeline

## Summary

Successfully downloaded the HamdiJr/Egyptian_hieroglyphs dataset!

**Next Steps**:
1. Parse the lexicon to create Gardiner → Transliteration mapping
2. Extract ResNet-50 features from the hieroglyph images
3. Build the fusion pipeline to combine text + visual embeddings